# Healthcare Challenge 1 - Baseline Submission

This notebook provides a simple baseline for **Healthcare Challenge 1: 30-Day Readmission Prediction**.

**Goal**: Predict `readmit_30d` (0/1) for each hospital admission
**Metric**: Macro-F1 Score - Higher is better

## Instructions:
1. **Replace API credentials** in the first cell with your team's API key and name
2. **Run all cells** to generate and submit baseline predictions
3. **Check the output** for your submission score

This baseline uses only tabular admission data with a simple Random Forest classifier.


In [ ]:
# 1. Initialize Client and Load Data

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from agentds import BenchmarkClient

# 🔑 REPLACE WITH YOUR CREDENTIALS
client = BenchmarkClient(
    api_key="your-api-key-here",        # Get from your team dashboard
    team_name="your-team-name-here"     # Your exact team name
)

# Load data from PVC paths
print("📂 Loading Healthcare Challenge 1 data...")

# Load admission data
train_admissions = pd.read_csv("/home/jovyan/shared/datasets/Healthcare/admissions_train.csv")
test_admissions = pd.read_csv("/home/jovyan/shared/datasets/Healthcare/admissions_test.csv")

print(f"✅ Data loaded:")
print(f"   Train admissions: {train_admissions.shape}")
print(f"   Test admissions: {test_admissions.shape}")
print(f"   Train columns: {list(train_admissions.columns)}")
print(f"   Test columns: {list(test_admissions.columns)}")


In [ ]:
# 2. Tabular-Only Baseline Model and Predictions

# From data inspection - admissions columns:
# admission_id, patient_id, primary_dx, los_days, acuity_emergent, charlson_band, ed_visits_6m, discharge_weekday, readmit_30d (train only)

# Select numeric features for baseline
admission_features = ['los_days', 'acuity_emergent', 'charlson_band', 'ed_visits_6m', 'discharge_weekday']
print(f"📊 Using admission features: {admission_features}")

# Prepare training data
X_train = train_admissions[admission_features].fillna(0)
y_train = train_admissions['readmit_30d']  # Binary target (0/1)

# Prepare test data
X_test = test_admissions[admission_features].fillna(0)

# Train simple Random Forest baseline
print("🤖 Training Random Forest classifier...")
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Make predictions
predictions = model.predict(X_test)

# Create submission file (format: admission_id,readmit_30d)
submission_df = pd.DataFrame({
    'admission_id': test_admissions['admission_id'],
    'readmit_30d': predictions
})

# Save predictions
submission_df.to_csv("healthcare_challenge1_predictions.csv", index=False)
print(f"✅ Predictions saved: {submission_df.shape[0]} predictions")
print(f"   Preview: {submission_df.head(3)}")
print(f"   Readmission rate: {predictions.mean():.3f} ({predictions.sum()} readmissions out of {len(predictions)})")


In [ ]:
# 3. Submit Predictions

# Submit predictions to the competition
print("🚀 Submitting predictions...")

try:
    result = client.submit_prediction("Healthcare", 1, "healthcare_challenge1_predictions.csv")
    
    if result['success']:
        print("✅ Submission successful!")
        print(f"   📊 Score: {result['score']:.4f}")
        print(f"   📏 Metric: {result['metric_name']}")
        print(f"   ✔️  Validation: {'Passed' if result['validation_passed'] else 'Failed'}")
    else:
        print("❌ Submission failed!")
        print(f"   Error details: {result.get('details', {}).get('validation_errors', 'Unknown error')}")
        
except Exception as e:
    print(f"💥 Submission error: {e}")
    print("🔧 Check your API key and team name are correct!")

print("\n🎯 Next steps:")
print("   1. Try incorporating relevant information outside this table!")
print("   2. Move on to Healthcare Challenge 2!")
